In [ ]:
## Running for mRNA-involved interactions
## Files to use:
# - gene2refseq.gz
# - Mus_musculus/{mouse.1.rna.fn.gz,mouse.2.rna.fn.gz,mouse.3.rna.fn.gz}

In [24]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from ete3 import NCBITaxa
import gzip
import re
from Bio import SeqIO

import time
from Bio import Entrez, SeqIO
from io import StringIO
from collections import deque


In [2]:
# Upload the dataset, maybe you have in a different way this
DATA_DIR = Path("RNAInter_RR_extracted")
txt_files = list(DATA_DIR.glob("*.txt"))
DATA_PATH = txt_files[0]
df = pd.read_csv(DATA_PATH, sep="\t", dtype=str, low_memory=False)

In [3]:
# This are the same conditions as non-coding RNA
exclude_categories = {
    'Category1', 'Category2', 'unknown', 'unassigned RNA', 'others',
    'Protein', 'protein',
    'PCG',
    'TF',
    'TR_C_gene',
    'antisense',
    'TEC',
    'repeats'
}

cat_cols = ['Category1', 'Category2']
mask_exclude = df[cat_cols].isin(exclude_categories).any(axis=1)
df_filtered = df[~mask_exclude].copy()

print("Rows before:", len(df))
print("Rows after :", len(df_filtered))

# Remove rows with NaN in Raw_ID1 or Raw_ID2
df_filtered = df_filtered.dropna(subset=['Raw_ID1', 'Raw_ID2'])
print("Rows after NaN removal :", len(df_filtered))

Rows before: 9484609
Rows after : 9429474
Rows after NaN removal : 8976390


In [4]:
parsed = []

def parse_ids(id_value):
    if pd.isna(id_value):
        return [None]
    elif ':' in id_value:
        prefix, ids_str = id_value.split(':', 1)
        return [f"{prefix}:{id_.strip()}" for id_ in ids_str.split(';')]
    else:
        parsed.append(id_value)
        return [id_value.strip()]

# Add list columns
df_filtered['Raw_ID1_list'] = df_filtered['Raw_ID1'].apply(parse_ids)
df_filtered['Raw_ID2_list'] = df_filtered['Raw_ID2'].apply(parse_ids)

df_filtered.replace({'-': None}, inplace=True)
unique_species = pd.unique(df_filtered[['Species1', 'Species2']].values.ravel())
print("Unique species:", len(unique_species))

ncbi = NCBITaxa()
unique_species_taxids = {}
missing = []

for test_species in unique_species:
    # Skip placeholders
    if test_species in [None]:
        continue

    # Handle known special cases first
    if test_species == 'Solanum tuberosum L.':
        unique_species_taxids[test_species] = "4113"
        continue
    if test_species == 'Rhodobacter sphaeroides ATCC 17025':
        unique_species_taxids[test_species] = "1061"
        continue

    # Query NCBI
    try:
        taxid = str(ncbi.get_name_translator([test_species])[test_species][0])
        unique_species_taxids[test_species] = taxid
    except KeyError:
        missing.append(test_species)

print("Mapped species (unique -1 for None):", len(unique_species_taxids))
print("Missing species:", missing)

df_filtered['Species1_TaxID'] = df_filtered['Species1'].map(unique_species_taxids)
df_filtered['Species2_TaxID'] = df_filtered['Species2'].map(unique_species_taxids)

unmapped_s1 = df_filtered.loc[df_filtered['Species1'].notna() & df_filtered['Species1_TaxID'].isna(), 'Species1'].unique()
unmapped_s2 = df_filtered.loc[df_filtered['Species2'].notna() & df_filtered['Species2_TaxID'].isna(), 'Species2'].unique()

print("Unmapped Species1:", unmapped_s1)
print("Unmapped Species2:", unmapped_s2)

print(f"RNA–RNA interactions: {len(df_filtered)}")
df_filtered = df_filtered.dropna(subset=['Species1_TaxID', 'Species2_TaxID']) # removing NAN
print(f"RNA–RNA interactions: {len(df_filtered)}")

Unique species: 98
Mapped species (unique -1 for None): 97
Missing species: []
Unmapped Species1: []
Unmapped Species2: []
RNA–RNA interactions: 8976390
RNA–RNA interactions: 8976389


### mRNA cases

In [5]:
# ------------------------- Extract mRNA-involved interactions and entities -------------------------
df_mrna = df_filtered[(df_filtered["Category1"] == "mRNA") | (df_filtered["Category2"] == "mRNA")].copy()
print("mRNA-involved interactions:", len(df_mrna))

def normalize_id_list(x):
    if isinstance(x, list):
        return tuple(sorted(str(i) for i in x if i is not None))
    return tuple()

df_mrna['Raw_ID1_key'] = df_mrna['Raw_ID1_list'].apply(normalize_id_list)
df_mrna['Raw_ID2_key'] = df_mrna['Raw_ID2_list'].apply(normalize_id_list)

# Only take entities where Category == mRNA
mrna_entities_1 = df_mrna[df_mrna["Category1"] == "mRNA"][['Interactor1.Symbol', 'Category1', 'Species1_TaxID', 'Raw_ID1_key']]
mrna_entities_2 = df_mrna[df_mrna["Category2"] == "mRNA"][['Interactor2.Symbol', 'Category2', 'Species2_TaxID', 'Raw_ID2_key']]

mrna_entities = (
    pd.concat([
        mrna_entities_1.rename(columns={
            'Interactor1.Symbol': 'Symbol',
            'Category1': 'Category',
            'Species1_TaxID': 'Species',
            'Raw_ID1_key': 'Raw_ID_key'
        }),
        mrna_entities_2.rename(columns={
            'Interactor2.Symbol': 'Symbol',
            'Category2': 'Category',
            'Species2_TaxID': 'Species',
            'Raw_ID2_key': 'Raw_ID_key'
        })], ignore_index=True).drop_duplicates())

print("Unique mRNA entities:", len(mrna_entities))

mRNA-involved interactions: 8786144
Unique mRNA entities: 184913


In [6]:
# Looking for the number of unique species in the mRNA entities
print("Number of Species:", mrna_entities['Species'].nunique())
# Analyze the sources of the mRNA entities (ARE NCBIs MOST COMMON)
sources = (mrna_entities["Raw_ID_key"].astype(str).str.replace(r"[(),]", "", regex=True).str.split(":").str[0])
source_counts = sources.value_counts(dropna=False)
print(source_counts)

Number of Species: 86
Raw_ID_key
'NCBI       183796
'Ensembl      1115
"NCBI            2
Name: count, dtype: int64


In [7]:
print("Top 5 Species by count:")
print(mrna_entities['Species'].value_counts().head(5))

Top 5 Species by count:
Species
10090    22747
7955     21950
9606     20837
6239     19970
10116    16410
Name: count, dtype: int64


# VERSION 2: ALL SPECIES

In [28]:
# In this version what I do is just taken directly with the API of NCBI, just to see how work. 

In [9]:
# -----------------------------
# CONFIG
# -----------------------------
Entrez.email = "ferreirasantiago01@gmail.com"          # required by NCBI
Entrez.api_key = "1a046d6a25e85a415626c363bd02c134a008"    
BATCH_SIZE = 200          # safe default
SLEEP_SECONDS = 0.34      # ~3 req/sec without api_key. If you set api_key you can reduce.
MAX_RETRIES = 5

In [13]:
df = mrna_entities.copy()
s = df["Raw_ID_key"].astype(str)
mask_ncbi = s.str.contains(r"NCBI:\s*\d+", regex=True) # keep only rows that truly contain an NCBI GeneID
df = df[mask_ncbi].copy()
s = s[mask_ncbi]

# extract GeneID
df["GeneID"] = (
    pd.to_numeric(
        s.str.replace(r"[()'\",]", "", regex=True) .str.extract(r"NCBI:\s*(\d+)", expand=False), errors="coerce").astype("Int64"))

In [14]:
# -----------------------------
# 2) Load gene2refseq.gz and map (tax_id, GeneID) -> best RefSeq RNA accession
#    (Here we prioritize curated mRNA NM_; you can expand to XM_ if desired.)
# -----------------------------
g2r = pd.read_csv(
    "gene2refseq.gz",
    sep="\t",
    comment="#",
    header=None,
    dtype=str,
    low_memory=False,
)

In [15]:
# Columns we need: tax_id, GeneID, status, RNA accession.version
g2r = g2r[[0, 1, 2, 3]].rename(columns={0: "tax_id", 1: "GeneID", 2: "status", 3: "rna_acc_ver"})

# Filter to mRNA accessions (NM_). If you want predicted too, include XM_.
g2r = g2r[
    g2r["rna_acc_ver"].notna()
    & (g2r["rna_acc_ver"] != "-")
    & (g2r["rna_acc_ver"].str.startswith("NM_"))
].copy()

# Prefer higher-quality status if present
status_rank = {"REVIEWED": 0, "VALIDATED": 1, "PROVISIONAL": 2, "INFERRED": 3, "PREDICTED": 4, "MODEL": 5}
g2r["rank"] = g2r["status"].map(status_rank).fillna(999).astype(int)

# Pick best per (tax_id, GeneID)
g2r_best = (
    g2r.sort_values(["tax_id", "GeneID", "rank", "rna_acc_ver"])
       .drop_duplicates(["tax_id", "GeneID"], keep="first")
)

In [20]:
df["GeneID"] = df["GeneID"].astype(str)
g2r_best["GeneID"] = g2r_best["GeneID"].astype(str)

df = df.merge(
    g2r_best[["tax_id", "GeneID", "rna_acc_ver"]],
    left_on=["Species", "GeneID"],
    right_on=["tax_id", "GeneID"],
    how="left"
)
df = df.rename(columns={"rna_acc_ver": "RefSeq"}).drop(columns=["tax_id"])
df

,Symbol,Category,Species,Raw_ID_key,GeneID,RefSeq
0,0610009B22Rik,mRNA,10090,"(NCBI:66050,)",66050,NM_001417819.1
1,0610010F05Rik,mRNA,10090,"(NCBI:71675,)",71675,NM_001347242.1
2,0610010K14Rik,mRNA,10090,"(NCBI:104457,)",104457,NM_001177601.1
3,0610012D04Rik,mRNA,10090,"(NCBI:71676,)",71676,NaN
4,0610012G03Rik,mRNA,10090,"(NCBI:106264,)",106264,NM_134107.1
...,...,...,...,...,...,...
183735,zym1,mRNA,4896,"(NCBI:2541743,)",2541743,NM_001019174.3
183736,zyx,mRNA,7955,"(NCBI:564246,)",564246,NM_001109824.2
183737,zzef1,mRNA,7955,"(NCBI:566682,)",566682,NaN
183738,zzef1,mRNA,8364,"(NCBI:100487543,)",100487543,NaN


In [22]:
# How many mRNA entities got a RefSeq mapping?
df["RefSeq"].notna().sum()

139157

In [26]:
# -----------------------------
# 3) Batch-fetch FASTA sequences by RefSeq accession (unique across all species)
# -----------------------------
def batched(iterable, n):
    it = list(iterable)
    for i in range(0, len(it), n):
        yield it[i:i+n]

def efetch_fasta(accessions):
    """Fetch FASTA for a list of nucleotide accessions."""
    # NCBI expects comma-separated IDs
    ids = ",".join(accessions)
    handle = Entrez.efetch(db="nuccore", id=ids, rettype="fasta", retmode="text")
    txt = handle.read()
    handle.close()
    return txt

unique_acc = df["RefSeq"].dropna().unique().tolist()

refseq_to_seq = {}
for batch in batched(unique_acc, BATCH_SIZE):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            fasta_txt = efetch_fasta(batch)
            for rec in SeqIO.parse(StringIO(fasta_txt), "fasta"):
                # rec.id can include version; keep exact match.
                refseq_to_seq[rec.id] = str(rec.seq)
            break
        except Exception as e:
            if attempt == MAX_RETRIES:
                print(f"[FAIL] Batch starting {batch[0]}: {e}")
            else:
                time.sleep(2 ** attempt * 0.5)
    time.sleep(SLEEP_SECONDS)

# Assign sequences
df["Sequence"] = df["RefSeq"].map(refseq_to_seq)

print("Matched RefSeq:", df["RefSeq"].notna().sum(), "/", len(df))
print("Matched sequences:", df["Sequence"].notna().sum(), "/", len(df))

Matched RefSeq: 139157 / 183740
Matched sequences: 139157 / 183740


In [27]:
df

,Symbol,Category,Species,Raw_ID_key,GeneID,RefSeq,Sequence
0,0610009B22Rik,mRNA,10090,"(NCBI:66050,)",66050,NM_001417819.1,ATTTTGCTTTCTGGCATTCTTGTAGTTAAAAGCTCCGCCTCACTTG...
1,0610010F05Rik,mRNA,10090,"(NCBI:71675,)",71675,NM_001347242.1,AGGCGGGTGCGGCCGCGGCTCGTGAGGCCGCCTCAGCCTGGCGTGT...
2,0610010K14Rik,mRNA,10090,"(NCBI:104457,)",104457,NM_001177601.1,GGGACTTCCTGCGCGGGAGCCCGAGCGGCCGGCTGCCGGGCTCTCC...
3,0610012D04Rik,mRNA,10090,"(NCBI:71676,)",71676,NaN,NaN
4,0610012G03Rik,mRNA,10090,"(NCBI:106264,)",106264,NM_134107.1,GTGGCCGTCCGGAAGGCGAAGGCAGCGAGCGTCGGGTTCCGTGGCG...
...,...,...,...,...,...,...,...
183735,zym1,mRNA,4896,"(NCBI:2541743,)",2541743,NM_001019174.3,ACTTTAAGCGTGCTTGAGACAATCGACTTTCAACAACTCATATCAC...
183736,zyx,mRNA,7955,"(NCBI:564246,)",564246,NM_001109824.2,AGATTCACTGGTTGACGTCCCCTCTAGAGCTTCCTAGTTTTAACAG...
183737,zzef1,mRNA,7955,"(NCBI:566682,)",566682,NaN,NaN
183738,zzef1,mRNA,8364,"(NCBI:100487543,)",100487543,NaN,NaN


In [ ]:
# 75% (139.157/183740) of all the mRNA entities we can retrieve if sequence data from RefSeq in 35 min. Not Bad! 

# VERSION 1: JUST MUS MUSCULUS 

In [ ]:
# Lets try to get the sequences for the Mus Musculus (Species 10090)
# Retrieve the sequences from: https://ftp.ncbi.nlm.nih.gov/refseq/M_musculus/mRNA_Prot/ 
# To get (mouse.1.rna.fna.gz, mouse.2.rna.fna.gz, mouse.3.rna.fna.gz)
# And mapping Gene to RefSeq file: [https://ftp.ncbi.nlm.nih.gov/gene/DATA/gene2refseq.gz](https://ftp.ncbi.nlm.nih.gov/gene/DATA/gene2refseq.gz) 

In [11]:
# -----------------------------
# 1) Parse GeneID from Raw_ID_key with only NCBI IDs
# -----------------------------
mrna_entities_10090 = mrna_entities[mrna_entities["Species"].astype(str) == "10090"].copy()
mrna_entities_10090["GeneID"] = (mrna_entities_10090["Raw_ID_key"].astype(str).str.extract(r"NCBI:(\d+)", expand=False))
mrna_entities_10090["GeneID"] = pd.to_numeric(mrna_entities_10090["GeneID"], errors="coerce").astype("Int64")

In [12]:
sources = (mrna_entities_10090["Raw_ID_key"].astype(str).str.replace(r"[(),]", "", regex=True).str.split(":").str[0])
source_counts = sources.value_counts(dropna=False)
print(source_counts)

Raw_ID_key
'NCBI       22595
'Ensembl      152
Name: count, dtype: int64


In [ ]:
## Just for double checking that all the GeneIDs are correctly parsed and come from NCBI
# -----------------------------
mrna_entities_10090 = mrna_entities[mrna_entities["Species"].astype(str) == "10090"].copy()
s = mrna_entities_10090["Raw_ID_key"].astype(str)
mask_ncbi = s.str.contains(r"NCBI:\s*\d+", regex=True) # keep only rows that truly contain an NCBI GeneID
mrna_entities_10090 = mrna_entities_10090[mask_ncbi].copy()
s = s[mask_ncbi]

# extract GeneID
mrna_entities_10090["GeneID"] = (
    pd.to_numeric(
        s.str.replace(r"[()'\",]", "", regex=True) 
         .str.extract(r"NCBI:\s*(\d+)", expand=False),
        errors="coerce"
    ).astype("Int64")
)

mrna_entities_10090
# So 22595 rows are NCBI GeneIDs for Mus Musculus, which is a good number to work with for the next steps.

,Symbol,Category,Species,Raw_ID_key,GeneID
0,0610009B22Rik,mRNA,10090,"(NCBI:66050,)",66050
1,0610010F05Rik,mRNA,10090,"(NCBI:71675,)",71675
6,0610010K14Rik,mRNA,10090,"(NCBI:104457,)",104457
16,0610012D04Rik,mRNA,10090,"(NCBI:71676,)",71676
17,0610012G03Rik,mRNA,10090,"(NCBI:106264,)",106264
...,...,...,...,...,...
8597369,mt-Atp8,mRNA,10090,"(NCBI:17706,)",17706
8597370,mt-Co1,mRNA,10090,"(NCBI:17708,)",17708
8597371,mt-Co3,mRNA,10090,"(NCBI:17710,)",17710
8597372,mt-Nd5,mRNA,10090,"(NCBI:17721,)",17721


In [14]:
# -----------------------------
# 2) Load gene2refseq.gz and map GeneID 
# -----------------------------
gene2refseq_path = "gene2refseq.gz"


# From here is ChatGPT's code and Copilot, but it works, so I will keep it as is, and understand it later
# gene2refseq is tab-delimited; comment lines start with '#'
g2r = pd.read_csv(
    gene2refseq_path,
    sep="\t",
    comment="#",
    header=None,
    dtype=str,
    low_memory=False,
)
# 10 minutes to load (maybe can be faster)... we have the gene2refseq loaded 

In [15]:
# NCBI gene2refseq columns (relevant ones):
# 0 tax_id, 1 GeneID, 2 status and 3 RNA_nucleotide_accession.version (others exist; we only need these)
g2r = g2r[[0, 1, 2, 3]].rename(columns={0: "tax_id", 1: "GeneID", 2: "status", 3: "rna_acc_ver"})

# Filter mouse + RefSeq mRNA only (NM_)
g2r = g2r[
    (g2r["tax_id"] == "10090")
    & (g2r["rna_acc_ver"].notna())
    & (g2r["rna_acc_ver"].str.startswith("NM_"))
    & (g2r["rna_acc_ver"] != "-")].copy()

# Prefer higher-quality status if present
status_rank = {"REVIEWED": 0, "VALIDATED": 1, "PROVISIONAL": 2, "INFERRED": 3, "PREDICTED": 4, "MODEL": 5}
g2r["rank"] = g2r["status"].map(status_rank).fillna(999).astype(int)

# For each GeneID, pick the best NM_ (lowest rank). If ties, keep first.
g2r_best = (
    g2r.sort_values(["GeneID", "rank", "rna_acc_ver"])
       .drop_duplicates("GeneID", keep="first")
)

geneid_to_refseq = dict(zip(g2r_best["GeneID"], g2r_best["rna_acc_ver"]))
mrna_entities_10090["RefSeq"] = mrna_entities_10090["GeneID"].astype(str).map(geneid_to_refseq)

In [16]:
# -----------------------------
# 3) Extract sequences for only the needed RefSeq accessions from your FASTA .gz files
# -----------------------------
fasta_files = ["Mus_musculus/mouse.1.rna.fna.gz", "Mus_musculus/mouse.2.rna.fna.gz", "Mus_musculus/mouse.3.rna.fna.gz"]

needed = set(mrna_entities_10090["RefSeq"].dropna().unique())
refseq_to_seq = {}

for fp in fasta_files:
    with gzip.open(fp, "rt") as handle:
        for record in SeqIO.parse(handle, "fasta"):
            acc = record.id  # e.g., NM_001... (no version if FASTA uses .1, it will be in id)
            if acc in needed:
                refseq_to_seq[acc] = str(record.seq)

mrna_entities_10090["Sequence"] = mrna_entities_10090["RefSeq"].map(refseq_to_seq)

# -----------------------------
# Done: mrna_entities_10090 now has RefSeq + Sequence
# -----------------------------
print(mrna_entities_10090[["Symbol", "Raw_ID_key", "GeneID", "RefSeq", "Sequence"]].head())
print("Matched RefSeq:", mrna_entities_10090["RefSeq"].notna().sum(), "/", len(mrna_entities_10090))
print("Matched sequences:", mrna_entities_10090["Sequence"].notna().sum(), "/", len(mrna_entities_10090))

           Symbol      Raw_ID_key  GeneID          RefSeq  \
0   0610009B22Rik   (NCBI:66050,)   66050  NM_001417819.1   
1   0610010F05Rik   (NCBI:71675,)   71675  NM_001347242.1   
6   0610010K14Rik  (NCBI:104457,)  104457  NM_001177601.1   
16  0610012D04Rik   (NCBI:71676,)   71676             NaN   
17  0610012G03Rik  (NCBI:106264,)  106264     NM_134107.1   

                                             Sequence  
0   ATTTTGCTTTCTGGCATTCTTGTAGTTAAAAGCTCCGCCTCACTTG...  
1   AGGCGGGTGCGGCCGCGGCTCGTGAGGCCGCCTCAGCCTGGCGTGT...  
6   GGGACTTCCTGCGCGGGAGCCCGAGCGGCCGGCTGCCGGGCTCTCC...  
16                                                NaN  
17  GTGGCCGTCCGGAAGGCGAAGGCAGCGAGCGTCGGGTTCCGTGGCG...  
Matched RefSeq: 21155 / 22595
Matched sequences: 21001 / 22595


In [17]:
mrna_entities_10090

,Symbol,Category,Species,Raw_ID_key,GeneID,RefSeq,Sequence
0,0610009B22Rik,mRNA,10090,"(NCBI:66050,)",66050,NM_001417819.1,ATTTTGCTTTCTGGCATTCTTGTAGTTAAAAGCTCCGCCTCACTTG...
1,0610010F05Rik,mRNA,10090,"(NCBI:71675,)",71675,NM_001347242.1,AGGCGGGTGCGGCCGCGGCTCGTGAGGCCGCCTCAGCCTGGCGTGT...
6,0610010K14Rik,mRNA,10090,"(NCBI:104457,)",104457,NM_001177601.1,GGGACTTCCTGCGCGGGAGCCCGAGCGGCCGGCTGCCGGGCTCTCC...
16,0610012D04Rik,mRNA,10090,"(NCBI:71676,)",71676,NaN,NaN
17,0610012G03Rik,mRNA,10090,"(NCBI:106264,)",106264,NM_134107.1,GTGGCCGTCCGGAAGGCGAAGGCAGCGAGCGTCGGGTTCCGTGGCG...
...,...,...,...,...,...,...,...
8597369,mt-Atp8,mRNA,10090,"(NCBI:17706,)",17706,NaN,NaN
8597370,mt-Co1,mRNA,10090,"(NCBI:17708,)",17708,NaN,NaN
8597371,mt-Co3,mRNA,10090,"(NCBI:17710,)",17710,NaN,NaN
8597372,mt-Nd5,mRNA,10090,"(NCBI:17721,)",17721,NaN,NaN


In [18]:
# So from 22595 rows are NCBI GeneIDs for Mus Musculus, we get 21001 sequences (~93%) and a little more RefSeq IDs.
# But this working just for Mus Musculus, we can try to do the same for other species, but in my case I need to downdoloaded the Mus_Musculus folder with the sequences
# Maybe theres a different way to get the sequences for other species without downloading all the FASTA files.